# 📊 Google GISLR Dataset Preprocessing Pipeline

This notebook implements the preprocessing pipeline for the **250-class Google Isolated Sign Language Recognition (GISLR)** dataset. 

### Pipeline Highlights:
1. **Uniform Sequence Interpolation**: Converts variable-frame parquets into fixed length sequence of exactly **30 frames**.
2. **Feature Slicing**: Slices Lips (40), Dominant Hand (21), and Pose (10) landmarks to produce 71 landmarks.
3. **Symmetry Mirroring**: Mirrors the dominant hand coordinate system into a canonical Left Hand layout.
4. **Unified Normalization (Phase 2)**: Centers hand coordinates around the wrist joint locally, and face/pose landmarks globally around the first lip landmark.
5. **Velocity Vectors**: Appends first-order difference features (`dx, dy, dz`) to produce features of shape `(30, 71, 6)`.
6. **Parallel Sharding**: Uses multi-threaded extraction (`ThreadPoolExecutor`) and saves preprocessed features in shards of **5,000 samples** to avoid Google Drive I/O bottlenecks and RAM exhaustion.

In [2]:
import os
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Define dataset directory (adjust to match your exact Drive paths)
DATA_DIR = "/content/drive/My Drive/Final Year Project/Dataset"
train_csv_path = os.path.join(DATA_DIR, "train.csv")
map_json_path = os.path.join(DATA_DIR, "sign_to_prediction_index_map.json")
OUTPUT_DIR = os.path.join(DATA_DIR, "preprocessed_kaggle")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load class mapping dictionary and training csv
with open(map_json_path, 'r', encoding='utf-8') as f:
    sign_map = json.load(f)
train_df = pd.read_csv(train_csv_path)
train_df['label'] = train_df['sign'].map(sign_map)

print(f"Total samples: {len(train_df):,}")
print(f"Vocabulary classes: {len(sign_map)}")

Total samples: 94,477
Vocabulary classes: 250


In [4]:
# Constants mapping MediaPipe indices (exactly aligned with our WLASL pipeline!)
LIPS_IDXS = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 146, 91, 181, 84, 17, 314, 405, 321, 375, 78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308]
LEFT_HAND_IDXS = np.arange(468, 489)
RIGHT_HAND_IDXS = np.arange(522, 543)
LEFT_POSE_IDXS = [500, 502, 504, 506, 508]
RIGHT_POSE_IDXS = [501, 503, 505, 507, 509]

def resize_sequence(frames_arr, target_length=30):
    """Resamples variable length frames to a fixed target length of 30 frames."""
    total_frames = len(frames_arr)
    indices = np.linspace(0, total_frames - 1, target_length).astype(int)
    return frames_arr[indices]

def preprocess_single_parquet(pq_path):
    """Preprocesses a single Kaggle parquet landmark file into (30, 71, 6) array."""
    try:
        # 1. Load coordinates (x, y, z)
        data = pd.read_parquet(pq_path, columns=['x', 'y', 'z'])
        n_frames = int(len(data) / 543)
        data = data.values.reshape(n_frames, 543, 3) # Shape: (Frames, 543, 3)
        
        # 2. Resample sequence length uniformly to 30 frames
        data = resize_sequence(data, target_length=30) # Shape: (30, 543, 3)
        
        # 3. Identify Dominant Hand based on presence of coordinates (non-NaN)
        lh_exists = np.sum(~np.isnan(data[:, LEFT_HAND_IDXS, 0]))
        rh_exists = np.sum(~np.isnan(data[:, RIGHT_HAND_IDXS, 0]))
        left_dominant = lh_exists >= rh_exists
        
        # 4. Extract landmarks and mirror if needed (converts everything to Left Hand orientation)
        lips = np.copy(data[:, LIPS_IDXS, :])
        if left_dominant:
            hand_1 = np.copy(data[:, LEFT_HAND_IDXS, :])
            pose = np.copy(data[:, LEFT_POSE_IDXS + RIGHT_POSE_IDXS, :])
        else: 
            hand_1 = np.copy(data[:, RIGHT_HAND_IDXS, :]) * [-1, 1, 1]
            pose = np.copy(data[:, RIGHT_POSE_IDXS + LEFT_POSE_IDXS, :]) * [-1, 1, 1]
            lips = lips * [-1, 1, 1]
            
        # 5. Local Hand Centering: Centering hand coordinates relative to wrist joint
        hand_1 = hand_1 - hand_1[:, 0:1, :]
        
        # 6. Global Face Centering: Centering lips and pose relative to the first lip coordinate
        face_ref = lips[:, 0:1, :]
        lips = lips - face_ref
        pose = pose - face_ref
        
        # 7. Concatenate features -> shape (30, 71, 3)
        clean_data = np.concatenate([lips, hand_1, pose], axis=1)
        clean_data = np.nan_to_num(clean_data, nan=0.0)
        
        # 8. Compute first-order temporal velocity -> shape (30, 71, 3)
        diff_data = np.diff(clean_data, axis=0, prepend=clean_data[0:1, :, :])
        
        # 9. Concatenate position and velocity -> shape (30, 71, 6)
        features = np.concatenate([clean_data, diff_data], axis=-1)
        return features.astype(np.float32)
    except Exception as e:
        print(f"Error processing {pq_path}: {e}")
        return None

In [5]:
# Shard size (save features in chunks of 5,000 to manage RAM)
SHARD_SIZE = 5000
total_samples = len(train_df)

print(f"Starting parallel preprocessing into {OUTPUT_DIR}...")

for shard_idx in range(0, total_samples, SHARD_SIZE):
    shard_num = shard_idx // SHARD_SIZE
    features_path = os.path.join(OUTPUT_DIR, f"X_shard_{shard_num}.npy")
    labels_path = os.path.join(OUTPUT_DIR, f"y_shard_{shard_num}.npy")
    
    if os.path.exists(features_path) and os.path.exists(labels_path):
        print(f"\nShard {shard_num} already processed. Skipping...")
        continue
    
    shard_df = train_df.iloc[shard_idx : shard_idx + SHARD_SIZE]
    shard_features = []
    shard_labels = []
    
    print(f"\nProcessing Shard {shard_idx // SHARD_SIZE} (Samples {shard_idx} to {shard_idx + len(shard_df)})...")
    
    # Convert paths relative to Drive location
    paths = [os.path.join(DATA_DIR, path) for path in shard_df['path']]
    
    # Read Parquet files in parallel using multi-threading
    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(tqdm(executor.map(preprocess_single_parquet, paths), total=len(paths)))
        
    for res, label in zip(results, shard_df['label']):
        if res is not None:
            shard_features.append(res)
            shard_labels.append(label)
            
    # Save features and labels as compressed arrays
    np.save(os.path.join(OUTPUT_DIR, f"X_shard_{shard_idx // SHARD_SIZE}.npy"), np.array(shard_features))
    np.save(os.path.join(OUTPUT_DIR, f"y_shard_{shard_idx // SHARD_SIZE}.npy"), np.array(shard_labels))
    
print("All shards processed and saved successfully!")

Starting parallel preprocessing into /content/drive/My Drive/Final Year Project/Dataset/preprocessed_kaggle...

Shard 0 already processed. Skipping...

Shard 1 already processed. Skipping...

Shard 2 already processed. Skipping...

Shard 3 already processed. Skipping...

Shard 4 already processed. Skipping...

Shard 5 already processed. Skipping...

Shard 6 already processed. Skipping...

Shard 7 already processed. Skipping...

Shard 8 already processed. Skipping...

Shard 9 already processed. Skipping...

Shard 10 already processed. Skipping...

Shard 11 already processed. Skipping...

Shard 12 already processed. Skipping...

Shard 13 already processed. Skipping...

Shard 14 already processed. Skipping...

Shard 15 already processed. Skipping...

Shard 16 already processed. Skipping...

Shard 17 already processed. Skipping...

Processing Shard 18 (Samples 90000 to 94477)...


  0%|          | 0/4477 [00:00<?, ?it/s]

All shards processed and saved successfully!
